`Q10` Biblioteca de una especie dada

`Q11` Biblioteca de un reino dado

Se toma en cuenta que la especie y reino son parametros, asi, las bibliotecas se generan en base a una misma consulta

In [38]:
from src.consultas_c import exportar_biblioteca
from pymongo import MongoClient

db = MongoClient("mongodb://localhost:27017/")["panteon_prueba"]

print(exportar_biblioteca(db, "especie", "Abortiporus biennis"))
print(exportar_biblioteca(db, "reino", "Fungi"))

('library\\biblioteca_especie_Abortiporus_biennis.fasta', 1, 1)
('library\\biblioteca_reino_Fungi.fasta', 1559, 19797)


``Interpretacion Q10``

La base de datos permite reconstruir la biblioteca completa de [especie] sin tocar el FASTA original: se exportaron [n] 
secuencias con el formato >ID#CLASE/ORDEN/SUPERFAMILIA @Especie. El número de encabezados del archivo coincide exactamente 
con el conteo de la consulta, lo que demuestra que no se perdió ni se duplicó ninguna secuencia en la carga ni en la exportación. 
La especie es un parámetro, así que la misma función genera la biblioteca de cualquiera de las 2790 especies.

``Interpretacion Q11``

El reino [Fungi] agrupa [n_esp] especies y [n_seq] secuencias, que representan el [X] % de las 239 946 secuencias de la base.
 Se generó con la misma función de Q10 cambiando solo el parámetro campo, lo que confirma que el diseño con especie_id como 
 referencia permite construir bibliotecas de cualquier nivel taxonómico con una consulta mínima, sin recorrer el FASTA.

`Q12` consulta que usa $lookup para unir secuencias con la taxonomía de su especie.

Usamos `"$limit":5` para delimitar la cantidad de secuencias a unir con la taxonomía de su especie.

In [39]:
pipeline = [
    {"$limit": 5},
    {"$lookup": {"from": "especies", "localField": "especie_id",
                 "foreignField": "_id", "as": "esp"}},
    {"$unwind": "$esp"},
    {"$project": {"_id": 0, "pdb_id": 1, "te_orden": 1,
                  "especie": "$esp.especie", "reino": "$esp.reino",
                  "filo": "$esp.filo", "tax_clase": "$esp.tax_clase",
                  "tax_orden": "$esp.tax_orden", "familia": "$esp.familia"}},
]
list(db.secuencias.aggregate(pipeline))

[{'pdb_id': 'PDB00000001',
  'te_orden': 'HELITRON',
  'especie': 'Halicephalobus mephisto',
  'reino': 'Animalia',
  'filo': 'Nematoda',
  'tax_clase': 'Chromadorea',
  'tax_orden': 'Rhabditida',
  'familia': 'Panagrolaimidae'},
 {'pdb_id': 'PDB00000002',
  'te_orden': 'HELITRON',
  'especie': 'Halicephalobus mephisto',
  'reino': 'Animalia',
  'filo': 'Nematoda',
  'tax_clase': 'Chromadorea',
  'tax_orden': 'Rhabditida',
  'familia': 'Panagrolaimidae'},
 {'pdb_id': 'PumCon-1.11',
  'te_orden': 'SINE',
  'especie': 'Puma concolor',
  'reino': 'Animalia',
  'filo': 'Chordata',
  'tax_clase': 'Mammalia',
  'tax_orden': 'Carnivora',
  'familia': 'Felidae'},
 {'pdb_id': 'PumCon-1.141',
  'te_orden': 'TIR',
  'especie': 'Puma concolor',
  'reino': 'Animalia',
  'filo': 'Chordata',
  'tax_clase': 'Mammalia',
  'tax_orden': 'Carnivora',
  'familia': 'Felidae'},
 {'pdb_id': 'PumCon-1.14',
  'te_orden': 'SINE',
  'especie': 'Puma concolor',
  'reino': 'Animalia',
  'filo': 'Chordata',
  'tax_c

``Interpretacion Q12``

Cada TE queda unido a su reino, filo, clase, orden y familia mediante una sola referencia (especie_id). La taxonomía vive en una sola colección de 2790 documentos y no se repite en las 239 946 secuencias, de modo que corregir un dato taxonómico exige modificar un documento y no miles. El resultado demuestra que el modelo referenciado funciona: ninguna secuencia quedó sin especie (0 sin match en la carga).

`Q13`  Comparación por origen de los datos

In [40]:
pipeline = [
    {"$group": {"_id": "$especie_id", "n_seq": {"$sum": 1},
                "suma_long": {"$sum": "$longitud"}}},
    {"$lookup": {"from": "especies", "localField": "_id",
                 "foreignField": "_id", "as": "esp"}},
    {"$unwind": "$esp"},
    {"$group": {"_id": "$esp.fuente",
                "n_especies": {"$sum": 1},
                "n_secuencias": {"$sum": "$n_seq"},
                "suma_long": {"$sum": "$suma_long"}}},
    {"$project": {"_id": 0, "fuente": "$_id", "n_especies": 1, "n_secuencias": 1,
                  "seq_por_especie": {"$divide": ["$n_secuencias", "$n_especies"]},
                  "longitud_media": {"$divide": ["$suma_long", "$n_secuencias"]}}},
]
import pandas as pd
pd.DataFrame(db.secuencias.aggregate(pipeline))

,n_especies,n_secuencias,fuente,seq_por_especie,longitud_media
0,2404,208354,RM2+MCH,86.669717,4375.507492
1,386,31592,Dfam,81.844560,1653.802450


`Interpretacion Q13`

Las especies con biblioteca Dfam tienen en promedio ``81.844`` secuencias, frente a `86.669` de RM2+MCH. Es esperable: Dfam es una base curada manualmente y consolidada, mientras que RM2+MCH produce de forma automática muchas familias redundantes o fragmentadas. Las longitudes medias (``4375.50`` pb frente a `1653.80` pb) indican que [las secuencias automáticas son más cortas, por fragmentación / ambos grupos tienen consensos de tamaño comparable]. Las dos fuentes no son equivalentes: al comparar especies entre grupos hay que tener en cuenta su origen.

`Q14` control de calidad 

In [49]:
pipeline = [
    {"$match": {"k": 1}},
    {"$lookup": {"from": "secuencias", "localField": "secuencia_id",
                 "foreignField": "_id", "as": "s"}},
    {"$unwind": "$s"},
    {"$match": {"$expr": {"$lt": ["$total_ventanas", "$s.longitud"]}}},
    {"$group": {"_id": None,
                "secuencias_afectadas": {"$sum": 1},
                "con_N": {"$sum": {"$cond": [{"$gt": ["$s.porcentaje_n", 0]}, 1, 0]}},
                "caracteres_descartados": {"$sum": {"$subtract": ["$s.longitud", "$total_ventanas"]}}}},
]
res = list(db.kmers.aggregate(pipeline))

total_seq = db.secuencias.count_documents({})
total_bases = next(db.secuencias.aggregate(
    [{"$group": {"_id": None, "bases": {"$sum": "$longitud"}}}]))["bases"]

if not res:
    print("Ninguna secuencia tiene caracteres distintos de ACGT.")
else:
    r = res[0]
    N = r["secuencias_afectadas"]
    X = N / total_seq * 100
    M = N - r["con_N"]
    Z = r["caracteres_descartados"]
    W = Z / total_bases * 100

    print(f"N = {N} secuencias afectadas (de {total_seq})")
    print(f"X = {X:.3f} %")
    print(f"M = {M} afectadas con porcentaje_n = 0 (IUPAC no detectados)")
    print(f"Z = {Z} caracteres descartados")
    print(f"W = {W:.5f} % de las {total_bases} bases")

N = 54837 secuencias afectadas (de 239946)
X = 22.854 %
M = 0 afectadas con porcentaje_n = 0 (IUPAC no detectados)
Z = 2147161 caracteres descartados
W = 0.22276 % de las 963901415 bases


`Interpretacion Q14`

54837 secuencias (22.85 %) contienen caracteres distintos de ACGT, y 0 de ellas tienen porcentaje_n = 0: son códigos IUPAC que ese campo no detecta. Es decir, porcentaje_n subestima los caracteres ambiguos y no basta como único control de calidad. En total se descartaron 2147161 ventanas de k=1, una pérdida de 0.222 % del total, que es despreciable para los conteos de k-mers. La política de descartar toda ventana no ACGT queda justificada: no se inventan datos y las frecuencias siguen normalizadas por total_ventanas.

`Q15`  Recuperar el perfil de k-mers de una secuencia dada, para un k dado.

In [ ]:
import pandas as pd

def perfil_kmers(db, pdb_id, k):
    s = db.secuencias.find_one({"pdb_id": pdb_id}, {"_id": 1})
    d = db.kmers.find_one({"secuencia_id": s["_id"], "k": k})
    df = pd.DataFrame({"kmer": list(d["conteos"]), "conteo": list(d["conteos"].values())})
    df["frecuencia"] = df["conteo"] / d["total_ventanas"]
    return df.sort_values("conteo", ascending=False).reset_index(drop=True), d["total_ventanas"]

df, total = perfil_kmers(db, "PDB00000001", 3)
print(total, len(df), "de", 4**3)
df.head(10) 

1989 64 de 64


,kmer,conteo,frecuencia
0,AAA,144,0.072398
1,TTT,97,0.048768
2,AAT,90,0.045249
3,ATT,74,0.037205
4,TAT,70,0.035194
5,ATA,68,0.034188
6,TAA,66,0.033183
7,TTA,65,0.032680
8,GAA,50,0.025138
9,ATG,47,0.023630


`Interpretacion Q15`

El perfil 3-mer de PDB00000001 muestra que 64 de los 64 trinucleótidos posibles aparecen, con total de 1989 ventanas. Los más frecuentes ([AAA, TTT, AAT...]) están formados solo por A y T, algo coherente con su GC de 31,5 %: la composición de k-mers reproduce la composición de bases. Esto confirma que los conteos guardados son correctos y que la colección kmers sirve como base para comparar secuencias y órdenes de TE